# Handling Outliers — The Percentile Method

A third approach, after Z-score (assumes normal) and IQR (assumes nothing about shape, but is tied to quartiles specifically): just pick a **fixed percentile cutoff** directly — here, the bottom and top 1% of `Height` values are treated as outliers, whatever their actual values happen to be. The simplest of the three methods conceptually, and the most explicit about how much of the data gets affected (exactly 1% on each end, by construction).

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('weight-height.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'weight-height.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['Height'].describe()

## 2. Checking the Shape

*(`sns.distplot()`/`sns.boxplot()` need fixing for current seaborn — positional args, and `distplot` itself is removed.)*

In [ ]:
sns.histplot(df['Height'], kde=True)
plt.show()


In [ ]:
sns.boxplot(x=df['Height'])
plt.show()


## 3. Computing the Percentile Boundaries

`.quantile(0.99)` / `.quantile(0.01)` mark the top and bottom 1% directly — no assumption about normality (unlike Z-score) and no dependence on the IQR specifically (unlike the previous notebook).

In [ ]:
upper_limit = df['Height'].quantile(0.99)
upper_limit

In [ ]:
lower_limit = df['Height'].quantile(0.01)
lower_limit

## 4. Trimming

### ⚠️ Hardcoded Values Instead of the Computed Variables

`74.78` and `58.13` are typed in directly, rather than using the `upper_limit`/`lower_limit` variables already computed above — if the data ever changes and those limits shift, this line would silently keep using the old, now-wrong numbers. Using the variables directly avoids that entirely.

In [ ]:
new_df = df[(df['Height'] <= upper_limit) & (df['Height'] >= lower_limit)]


In [ ]:
new_df['Height'].describe()

In [ ]:
df['Height'].describe()

### Visual Before/After Comparison

In [ ]:
plt.figure(figsize=(16, 8))
plt.subplot(2, 2, 1)
sns.histplot(df['Height'], kde=True)
plt.title('Before trimming')

plt.subplot(2, 2, 2)
sns.boxplot(x=df['Height'])
plt.title('Before trimming')

plt.subplot(2, 2, 3)
sns.histplot(new_df['Height'], kde=True)
plt.title('After trimming')

plt.subplot(2, 2, 4)
sns.boxplot(x=new_df['Height'])
plt.title('After trimming')

plt.show()


## 5. Capping (Winsorization)

This technique — clipping values to a fixed percentile boundary instead of removing them — specifically has its own name: **Winsorization**.

In [ ]:
# Capping --> Winsorization
df['Height'] = np.where(df['Height'] >= upper_limit,
        upper_limit,
        np.where(df['Height'] <= lower_limit,
        lower_limit,
        df['Height']))

In [ ]:
df.shape

In [ ]:
df['Height'].describe()

### Visual Before/After Comparison — Capping

In [ ]:
plt.figure(figsize=(16, 4))
plt.subplot(1, 2, 1)
sns.histplot(df['Height'], kde=True)
plt.title('After capping')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Height'])
plt.title('After capping')
plt.show()


## Summary — All Three Outlier Methods Compared

| Method | Assumption | Boundary formula | Best for |
|---|---|---|---|
| **Z-score** | Roughly normal data | `mean ± 3*std` | Symmetric, bell-shaped columns (`cgpa`) |
| **IQR** | None about shape | `Q1/Q3 ± 1.5*IQR` | Skewed columns (`placement_exam_marks`) |
| **Percentile** | None | fixed `.quantile(p)` cutoffs | Simple, explicit control over exactly how much data is affected (`Height`) |

Each supports both **trimming** (drop the rows) and **capping/Winsorization** (clip values to the boundary, keeping every row) — the same trimming-vs-capping trade-off applies regardless of which boundary method is used.